# SEC 8-K Event Dataset — reproduce the benchmark tables

This notebook loads one event class, takes prices from a source **you** supply, runs the event study with
the exact parameters of our run, and compares the result with the shipped tables. Change `EVENT_CLASS`,
the price source, or `StudyConfig` fields to run your own variant. Everything here is historical; see
`RISK_NOTICE.md`.

In [ ]:
import pandas as pd
from sec_events import (StudyConfig, compare_with_shipped, list_event_classes, load_events, load_prices_csv_dir,
                        load_summary, load_table, run_event_study, spliced_benchmark)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

EVENT_CLASS = "item_2_06_impairment"      # one of list_event_classes()
print(list_event_classes())

In [ ]:
events = load_events(EVENT_CLASS, benchmark_subset=True)   # exactly the rows behind data/benchmarks/<class>/
run = load_table(EVENT_CLASS, "run")
cfg = StudyConfig.from_run(run)
print(len(events), "events;", run["title"]); print(cfg)
events.head()

## 1. Prices (bring your own)

Option A: a directory of `<TICKER>.csv` files with `date,close` (adjusted) exported from your vendor.
Option B (personal research only, Yahoo's terms): `load_prices_yfinance`. The benchmark candidates
(`URTH`, `ACWI`, `SPY`) are needed as well.

In [ ]:
tickers = sorted(set(events.ticker)) + list(run["benchmark_candidates"])

PRICE_DIR = "my_prices"                     # option A
prices = load_prices_csv_dir(PRICE_DIR, tickers)
if not prices:                               # option B
    from sec_events import load_prices_yfinance
    prices = load_prices_yfinance(tickers)

symbol, benchmark = spliced_benchmark(prices, run["benchmark_candidates"])
print(len(prices), "price series;", "benchmark", symbol, "from", benchmark.index[0].date() if benchmark is not None else None)
missing = sorted(set(events.ticker) - set(prices)); print(len(missing), "tickers without prices:", missing[:20])

## 2. Run the event study

In [ ]:
res = run_event_study(events, prices, benchmark, symbol, cfg)
cols = ["horizon", "trades", "mean_return_pct", "median_return_pct", "win_rate_pct", "benchmark_mean_return_pct",
        "mean_abnormal_return_pct", "abnormal_adj_ci_low_pct", "abnormal_adj_ci_high_pct", "placebo_mean_return_pct",
        "mean_excess_vs_placebo_pct", "ct_annualized_alpha_pct", "ct_t_stat"]
res.summary[cols].round(2)

In [ ]:
res.skipped.reason.value_counts()

## 3. Compare with the shipped tables

Same prices: single numbers match exactly, CIs within sampling noise. Other vendor or vintage: a few tenths of a point (see `docs/KNOWN_LIMITATIONS.md`).

The comparison only lines up when the events are the ones our run used, which is what `benchmark_subset=True` selects.

In [ ]:
cmp = compare_with_shipped(res.summary, load_summary(EVENT_CLASS))
cmp.pivot(index="metric", columns="horizon", values="diff").round(2)

## 4. Breakdowns and the calendar-time portfolio

In [ ]:
res.breakdown_by_year[res.breakdown_by_year.horizon == "1y"].set_index("year")[["trades", "mean_abnormal_pct", "mean_excess_vs_placebo_pct"]].round(1)

In [ ]:
res.breakdown_by_regime.round(2)

In [ ]:
ct = res.calendar_time["1y"]
try:
    import matplotlib.pyplot as plt
    ax = ((1 + ct.set_index("month_end")[["strategy_return", "benchmark_return"]]).cumprod()).plot(figsize=(9, 4), title=f"{EVENT_CLASS}: calendar-time portfolio vs benchmark, 1y windows")
    ax.set_ylabel("growth of 1")
except ImportError:
    print(ct.tail())

## 5. Your own variant

Edit the config and rerun. Per-trade rows are in `res.trades` for your own filters (by year, `event_session`, `index_name`, SIC).

In [ ]:
my_cfg = StudyConfig(horizons_years=(0.5, 1.0, 2.0), entry_lag_sessions=0, stock_cost_bps=10.0, seed=7)
mine = run_event_study(events, prices, benchmark, symbol, my_cfg)
mine.summary[["horizon", "trades", "mean_return_pct", "benchmark_mean_return_pct", "mean_abnormal_return_pct", "abnormal_adj_ci_low_pct", "abnormal_adj_ci_high_pct"]].round(2)

In [ ]:
# export your run in the shipped layout
import os; os.makedirs("my_run", exist_ok=True)
mine.summary.to_csv("my_run/summary.csv", index=False); mine.trades.to_csv("my_run/trades.csv", index=False)
mine.breakdown_by_year.to_csv("my_run/breakdown_by_year.csv", index=False)
print(sorted(os.listdir("my_run")))